# Very first script to run - OBS HSBD data only
## simple script to extract, clean, and prepare only the necessary data

SOME MANUAL HANDS ON REQUIRED!

2026-04-21 REVAMPED
- new: smiles cleaning pipeline included already here
- deduplication step available at the end of this notebook

## Scope:
- HSBD compartments handled here: air, soil, water, sediment.
- This is Step 1 in the HSBD curation flow.

## Optional:
Some missing SMILES not found by OPSIN can be manually curated (expert fixing of names).
See [0a1_2_optional_missing_hsbbd_data.ipynb](0a1_2_optional_missing_hsbbd_data.ipynb).

If this optional curation is skipped, that is fine.
BUT: enable the duplicate removal step in this notebook in that case (see end of notebook)!

Further pipeline details are documented in:
- ../0_1_data_clean_descr_calcs_db_build/readme.md
- ../processed_data/readme.md

In [32]:
import sys
from typing import List
import pandas as pd
from pandas import read_csv
import numpy as np
from pathlib import Path
import requests
from tqdm.notebook import tqdm

notebookdir = Path.cwd().parent
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders
from src.rdkit_tools import smiles_standardizer_msc

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [33]:
# set directories and filenames, load data
working_dir = Path.cwd().parent
raw_data_dir = working_dir / "raw_data"
output_dir = working_dir / "processed_data"

air_file = raw_data_dir / "hsbd_data" / "air_CAS_to_SMILES_output.csv"
soil_file = raw_data_dir / "hsbd_data" / "soil_CAS_to_SMILES_output.csv"
water_file = raw_data_dir / "hsbd_data" / "water_CAS_to_SMILES_output.csv"
sediment_file = raw_data_dir / "hsbd_data" / "sediment_CAS_to_SMILES_output.csv"

air_data_all = read_csv(air_file, index_col=0, header=0, sep=",")
soil_data_all = read_csv(soil_file, index_col=0, header=0, sep=",")
water_data_all = read_csv(water_file, index_col=0, header=0, sep=",")
sediment_data_all = read_csv(sediment_file, index_col=0, header=0, sep=",")

air_data = air_data_all[["Compound", "CAS", "T_half_air_days", "Canonical_smiles"]].copy()
soil_data = soil_data_all[["Compound", "CAS", "T_half_soil_days", "Canonical_smiles"]].copy()
water_data = water_data_all[["Compound", "CAS", "T_half_water_days", "Canonical_smiles"]].copy()
sediment_data = sediment_data_all[["Compound", "CAS", "T_half_sediment_days", "Canonical_smiles"]].copy()
# set columns to str except for T_half columns
for df in [air_data, soil_data, water_data, sediment_data]:
    for col in df.columns:
        if col.startswith("T_half"):
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            df[col] = df[col].astype(str)

In [34]:
# prepare dataframes and strip quotes, etc
def strip_quotes_from_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Strip leading and trailing quotes from all string entries in the DataFrame."""
    # exlude smiles column from stripping!
    for col in df.select_dtypes(include=["object", "string"]).columns:
        if col != "Canonical_smiles":
            df[col] = df[col].str.strip('"')
    return df


air_data = strip_quotes_from_dataframe(air_data)
soil_data = strip_quotes_from_dataframe(soil_data)
water_data = strip_quotes_from_dataframe(water_data)
sediment_data = strip_quotes_from_dataframe(sediment_data)


def strip_PCB_text_from_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Strip trailing '  (PCB <number>)' from compound names.
    Not perfect, but should work for most cases. Exclude smiles column from this operation just in case.
    """

    for col in df.select_dtypes(include=["object", "string"]).columns:
        if col != "Canonical_smiles":
            df[col] = df[col].str.replace(r" +\(PCB \d+\)", "", regex=True)
    return df


air_data = strip_PCB_text_from_dataframe(air_data)
soil_data = strip_PCB_text_from_dataframe(soil_data)
water_data = strip_PCB_text_from_dataframe(water_data)
sediment_data = strip_PCB_text_from_dataframe(sediment_data)

In [35]:
# simplify column names for half-life to T_half_days
air_data_all = air_data_all.rename(columns={"T_half_air_days": "T_half_days"})
soil_data_all = soil_data_all.rename(columns={"T_half_soil_days": "T_half_days"})
water_data_all = water_data_all.rename(columns={"T_half_water_days": "T_half_days"})
sediment_data_all = sediment_data_all.rename(columns={"T_half_sediment_days": "T_half_days"})

In [36]:
dataframe_list = [air_data, soil_data, water_data, sediment_data]

In [ ]:
def fetch_missing_entries(df: pd.DataFrame, column_name: str = "Canonical_smiles") -> list[str]:
    """Fetch list of compounds with missing entries in specified column.
    Checks for NaN, 'None' (string), and empty string "".
    """
    missing_mask = (
        df[column_name].isna()
        | (df[column_name].astype(str).str.lower() == "nan")
        | (df[column_name].astype(str).str.lower() == "none")
        | (df[column_name] == "")
    )
    missing_entries = df[missing_mask]["Compound"].tolist()
    return missing_entries


def print_missing_smiles_stats(df: pd.DataFrame, missing_df: list, label: str) -> None:
    print(
        f"Extracted {len(df)} {label} entries, with {len(missing_df)} missing SMILES = {len(missing_df) / len(df) * 100:.2f}%."
    )

In [38]:
missing_smiles_air: list[str] = fetch_missing_entries(air_data)
missing_smiles_soil: list[str] = fetch_missing_entries(soil_data)
missing_smiles_water: list[str] = fetch_missing_entries(water_data)
missing_smiles_sediment: list[str] = fetch_missing_entries(sediment_data)

missing_smiles_lists = [
    missing_smiles_air,
    missing_smiles_soil,
    missing_smiles_water,
    missing_smiles_sediment,
]
# it would have been better to make a dictionary to avoid having to update list
# but now "it's too late"

In [ ]:
# print missing SMILES stats
labels = ["air", "soil", "water", "sediment"]  # updated labels, but not executed with labels as of yet
for df, ml, label in zip(dataframe_list, missing_smiles_lists, labels):
    print_missing_smiles_stats(df, ml, label)

Extracted 562 air t_half entries, with 69 missing SMILES = 12.28%.
Extracted 693 air t_half entries, with 39 missing SMILES = 5.63%.
Extracted 757 air t_half entries, with 111 missing SMILES = 14.66%.
Extracted 360 air t_half entries, with 32 missing SMILES = 8.89%.


In [40]:
def fetch_OPSIN_smiles_from_name(iupac_name: str) -> str | None:
    """Fetch SMILES string from IUPAC name using OPSIN API.
    Returns SMILES string if found, else None."""
    smiles: str | None = None
    response = requests.get(f"https://opsin.ch.cam.ac.uk/opsin/{iupac_name}.json")
    # Parse the response
    if response.status_code == 200:
        data = response.json()
        smiles = data.get("smiles")
        print(f"SMILES for {iupac_name}: {smiles}")
        return smiles if smiles else None
    else:
        return None

In [41]:
def retrieve_and_update_smiles(df: pd.DataFrame, missing_smiles: list[str]) -> None:
    """Fetch missing SMILES for all compounds in the provided list and directly updates the DataFrame."""
    # iterate over missing SMILES and fetch them
    counter = 0
    for compound in tqdm(missing_smiles, desc="Fetching missing SMILES"):
        smiles = fetch_OPSIN_smiles_from_name(compound)
        if smiles:
            counter += 1
        df.loc[df["Compound"] == compound, "Canonical_smiles"] = smiles  # update the dataframe directly
    print(f"Found {counter} compounds of missing {len(missing_smiles)}.")

    # retrospec review: not the cleanest way to return the dfs, but since I do it in a loop...

In [42]:
# retrieve missing SMILES for all dataframes
for df, ml in zip(dataframe_list, missing_smiles_lists):
    retrieve_and_update_smiles(df, ml)
    # note: write and update the dataframes in place, so no need to return them

Fetching missing SMILES:   0%|          | 0/69 [00:00<?, ?it/s]

SMILES for 1,1'-biphenyl: C1(=CC=CC=C1)C1=CC=CC=C1
SMILES for 1,1'-biphenyl -4,4'-diamine: C1(=CC=C(C=C1)N)C1=CC=C(C=C1)N
SMILES for cyclo-octane: C1CCCCCCC1
SMILES for decachlorobiphenyl: ClC1=C(C(=C(C(=C1C1=C(C(=C(C(=C1Cl)Cl)Cl)Cl)Cl)Cl)Cl)Cl)Cl
SMILES for 3 chlorobiphenyl: ClC=1C=C(C=CC1)C1=CC=CC=C1
SMILES for 2,3,4-trichlorobiphenyl: ClC1=C(C=CC(=C1Cl)Cl)C1=CC=CC=C1
SMILES for 2,3,3',4',5'-pentachlorobiphenyl: ClC1=C(C=CC=C1Cl)C1=CC(=C(C(=C1)Cl)Cl)Cl
SMILES for 2,2' dichloro 4 methylbiphenyl: ClC1=C(C=CC(=C1)C)C1=C(C=CC=C1)Cl
SMILES for 3,3',4,4',5,5'-hexachlorobiphenyl: ClC=1C=C(C=C(C1Cl)Cl)C1=CC(=C(C(=C1)Cl)Cl)Cl
SMILES for 2,2',3,3',4,5,6-heptachlorobiphenyl: ClC1=C(C(=C(C(=C1Cl)Cl)Cl)Cl)C1=C(C(=CC=C1)Cl)Cl
SMILES for 1,2,3,4,6,7,8-heptchlorodibenzo-p-dioxin: ClC1=C(C(=C(C=2OC3=C(OC21)C=C(C(=C3Cl)Cl)Cl)Cl)Cl)Cl
SMILES for 2,2',3,3',4,5,5',6,6'-nonachlorobiphenyl: ClC1=C(C(=C(C(=C1Cl)Cl)Cl)Cl)C1=C(C(=CC(=C1Cl)Cl)Cl)Cl
SMILES for iso-propyl-4-methyl-benzene: C(C)(C)C1=CC=C(C=C1)C


Fetching missing SMILES:   0%|          | 0/39 [00:00<?, ?it/s]

SMILES for CRESOL: C1(=CC=CC=C1O)C
SMILES for DICHLOROPROPENE: ClC(=CC)Cl
SMILES for HEXABROMOCYCLODODECANE: BrC1(C(C(CCCCCCCCC1)(Br)Br)(Br)Br)Br
SMILES for NONYLPHENOL: C(CCCCCCCC)C1=C(C=CC=C1)O
SMILES for 1,1'-biphenyl: C1(=CC=CC=C1)C1=CC=CC=C1
SMILES for 1,1'-biphenyl -4,4'-diamine: C1(=CC=C(C=C1)N)C1=CC=C(C=C1)N
SMILES for cyclo-octane: C1CCCCCCC1
SMILES for decachlorobiphenyl: ClC1=C(C(=C(C(=C1C1=C(C(=C(C(=C1Cl)Cl)Cl)Cl)Cl)Cl)Cl)Cl)Cl
SMILES for 3 chlorobiphenyl: ClC=1C=C(C=CC1)C1=CC=CC=C1
SMILES for 2,3,4-trichlorobiphenyl: ClC1=C(C=CC(=C1Cl)Cl)C1=CC=CC=C1
SMILES for 2,3,3',4',5'-pentachlorobiphenyl: ClC1=C(C=CC=C1Cl)C1=CC(=C(C(=C1)Cl)Cl)Cl
SMILES for 2,2' dichloro 4 methylbiphenyl: ClC1=C(C=CC(=C1)C)C1=C(C=CC=C1)Cl
SMILES for 3,3',4,4',5,5'-hexachlorobiphenyl: ClC=1C=C(C=C(C1Cl)Cl)C1=CC(=C(C(=C1)Cl)Cl)Cl
SMILES for 2,2',3,3',4,5,6-heptachlorobiphenyl: ClC1=C(C(=C(C(=C1Cl)Cl)Cl)Cl)C1=C(C(=CC=C1)Cl)Cl
SMILES for 1,2,3,4,6,7,8-heptchlorodibenzo-p-dioxin: ClC1=C(C(=C(C=2OC3=C(OC21)C

Fetching missing SMILES:   0%|          | 0/111 [00:00<?, ?it/s]

SMILES for CRESOL: C1(=CC=CC=C1O)C
SMILES for DIPROPYLENE GLYCOL: CC(COC(C)CO)O
SMILES for HEXABROMOCYCLODODECANE: BrC1(C(C(CCCCCCCCC1)(Br)Br)(Br)Br)Br
SMILES for NONYLPHENOL: C(CCCCCCCC)C1=C(C=CC=C1)O
SMILES for SODIUM DODECYLBENZENESULFONATE: C(CCCCCCCCCCC)OS(=O)(=O)C1=CC=CC=C1.[Na]
SMILES for 1,1'-biphenyl: C1(=CC=CC=C1)C1=CC=CC=C1
SMILES for 1,1'-biphenyl -4,4'-diamine: C1(=CC=C(C=C1)N)C1=CC=C(C=C1)N
SMILES for cyclo-octane: C1CCCCCCC1
SMILES for decachlorobiphenyl: ClC1=C(C(=C(C(=C1C1=C(C(=C(C(=C1Cl)Cl)Cl)Cl)Cl)Cl)Cl)Cl)Cl
SMILES for 3 chlorobiphenyl: ClC=1C=C(C=CC1)C1=CC=CC=C1
SMILES for 2,3,4-trichlorobiphenyl: ClC1=C(C=CC(=C1Cl)Cl)C1=CC=CC=C1
SMILES for 2,3,3',4',5'-pentachlorobiphenyl: ClC1=C(C=CC=C1Cl)C1=CC(=C(C(=C1)Cl)Cl)Cl
SMILES for 2,2' dichloro 4 methylbiphenyl: ClC1=C(C=CC(=C1)C)C1=C(C=CC=C1)Cl
SMILES for 3,3',4,4',5,5'-hexachlorobiphenyl: ClC=1C=C(C=C(C1Cl)Cl)C1=CC(=C(C(=C1)Cl)Cl)Cl
SMILES for 2,2',3,3',4,5,6-heptachlorobiphenyl: ClC1=C(C(=C(C(=C1Cl)Cl)Cl)Cl)C1=C(C(=CC

Fetching missing SMILES:   0%|          | 0/32 [00:00<?, ?it/s]

SMILES for NONYLPHENOL: C(CCCCCCCC)C1=C(C=CC=C1)O
SMILES for 1,1'-biphenyl: C1(=CC=CC=C1)C1=CC=CC=C1
SMILES for 1,1'-biphenyl -4,4'-diamine: C1(=CC=C(C=C1)N)C1=CC=C(C=C1)N
SMILES for cyclo-octane: C1CCCCCCC1
SMILES for decachlorobiphenyl: ClC1=C(C(=C(C(=C1C1=C(C(=C(C(=C1Cl)Cl)Cl)Cl)Cl)Cl)Cl)Cl)Cl
SMILES for 3 chlorobiphenyl: ClC=1C=C(C=CC1)C1=CC=CC=C1
SMILES for 2,3,4-trichlorobiphenyl: ClC1=C(C=CC(=C1Cl)Cl)C1=CC=CC=C1
SMILES for 2,3,3',4',5'-pentachlorobiphenyl: ClC1=C(C=CC=C1Cl)C1=CC(=C(C(=C1)Cl)Cl)Cl
SMILES for 2,2' dichloro 4 methylbiphenyl: ClC1=C(C=CC(=C1)C)C1=C(C=CC=C1)Cl
SMILES for 3,3',4,4',5,5'-hexachlorobiphenyl: ClC=1C=C(C=C(C1Cl)Cl)C1=CC(=C(C(=C1)Cl)Cl)Cl
SMILES for 2,2',3,3',4,5,6-heptachlorobiphenyl: ClC1=C(C(=C(C(=C1Cl)Cl)Cl)Cl)C1=C(C(=CC=C1)Cl)Cl
SMILES for 1,2,3,4,6,7,8-heptchlorodibenzo-p-dioxin: ClC1=C(C(=C(C=2OC3=C(OC21)C=C(C(=C3Cl)Cl)Cl)Cl)Cl)Cl
SMILES for 2,2',3,3',4,5,5',6,6'-nonachlorobiphenyl: ClC1=C(C(=C(C(=C1Cl)Cl)Cl)Cl)C1=C(C(=CC(=C1Cl)Cl)Cl)Cl
Found 13 com

In [ ]:
# update the lists of missing SMILES after fetching
missing_smiles_air2: list[str] = fetch_missing_entries(air_data)
missing_smiles_soil2: list[str] = fetch_missing_entries(soil_data)
missing_smiles_water2: list[str] = fetch_missing_entries(water_data)
missing_smiles_sediment2: list[str] = fetch_missing_entries(sediment_data)
missing_smiles_lists2 = [
    missing_smiles_air2,
    missing_smiles_soil2,
    missing_smiles_water2,
    missing_smiles_sediment2,
]
# print updated missing SMILES stats
dataframe_list2 = [air_data, soil_data, water_data, sediment_data]
labels = ["air", "soil", "water", "sediment"]  # updated labels, but not executed with labels as of yet
for df, ml, label in zip(dataframe_list2, missing_smiles_lists2, labels):
    print_missing_smiles_stats(df, ml, label)

Extracted 562 air t_half entries, with 50 missing SMILES = 8.90%.
Extracted 693 air t_half entries, with 23 missing SMILES = 3.32%.
Extracted 757 air t_half entries, with 55 missing SMILES = 7.27%.
Extracted 360 air t_half entries, with 19 missing SMILES = 5.28%.


In [44]:
# write missing lists to files
missing_smiles_filenames = [
    output_dir / "missing" / "missing_smiles_air.txt",
    output_dir / "missing" / "missing_smiles_soil.txt",
    output_dir / "missing" / "missing_smiles_water.txt",
    output_dir / "missing" / "missing_smiles_sediment.txt",
]

missing_dir = output_dir / "missing"
missing_dir.mkdir(parents=True, exist_ok=True)

for filename, missing_smiles in zip(
    missing_smiles_filenames, [missing_smiles_air2, missing_smiles_soil2, missing_smiles_water2, missing_smiles_sediment2]
):
    with open(filename, "w") as f:
        f.writelines("\n".join(missing_smiles))

In [45]:
# standardize smiles - copied from database creation notebook, but with some minor changes
df2_dict = {
    "air": air_data,
    "soil": soil_data,
    "water": water_data,
    "sediment": sediment_data,
}
# get all "Canonical_smiles", created Canonical SMILES to ensure all are actually canonical
# this should have been done during earlier data processing, but just to be sure and then not have any issues later on
for df in df2_dict.values():
    print(f"Before standardization: {len(df)}")
    unique_smiles: List[str] = df["Canonical_smiles"].unique().tolist()
    canonical_smiles = {}
    for smi in unique_smiles:
        if smi and smi.lower() not in ["nan", "none"]:
            canonical_smiles[smi] = smiles_standardizer_msc(smi)[0]
        else:
            canonical_smiles[smi] = None
    df["Canonical_smiles"] = df["Canonical_smiles"].map(canonical_smiles)
    # df.dropna(subset=["Canonical_smiles"], inplace=True) # we want to keep missing entries for now, in contrast to later database construction
    df.reset_index(drop=True, inplace=True)
    print(f"After standardization: {len(df)}\n")

Before standardization: 562
After standardization: 562

Before standardization: 693
After standardization: 693

Before standardization: 757
After standardization: 757

Before standardization: 360
After standardization: 360



In [ ]:
# not really needed here (anymore)
print(f"Before dropping duplicates based on Smiles:")
print(f"Air data: {len(air_data)}")
print(f"Soil data: {len(soil_data)}")
print(f"Water data: {len(water_data)}")
print(f"Sediment data: {len(sediment_data)}")

# mkdir for missing data with duplicates for optional review
# might be useful to have data w duplicates for optional review; used by the optional notebook, for merging with missing before duplicate removal
(output_dir / "missing" / "_hsbd_tmp").mkdir(parents=True, exist_ok=True)
air_data.to_csv(output_dir / "missing" / "_hsbd_tmp" / "hsbd_air_w_duplicates_for_optional.tmp", index=False, sep="\t")
soil_data.to_csv(output_dir / "missing" / "_hsbd_tmp" / "hsbd_soil_w_duplicates_for_optional.tmp", index=False, sep="\t")
water_data.to_csv(output_dir / "missing" / "_hsbd_tmp" / "hsbd_water_w_duplicates_for_optional.tmp", index=False, sep="\t")
sediment_data.to_csv(
    output_dir / "missing" / "_hsbd_tmp" / "hsbd_sediment_w_duplicates_for_optional.tmp", index=False, sep="\t"
)

Before dropping duplicates based on Smiles:
Air data: 562
Soil data: 693
Water data: 757
Sediment data: 360


# OBS - DO NOT REMOVE DUPLICATES IF YOU INTEND FINDING MISSING SMILES AND COMBINING IN NEXT STEP

In [ ]:
# def deduplicate_and_count_sources(df: pd.DataFrame, smiles_col: str = "Canonical_smiles") -> pd.DataFrame:
#     counts = df.groupby(smiles_col).size().rename("n_sources")
#     df = (
#         df.drop_duplicates(subset=smiles_col)
#           .merge(counts, on=smiles_col)
#     )
#     return df

# air_before_dedup = len(air_data)
# soil_before_dedup = len(soil_data)
# water_before_dedup = len(water_data)
# sediment_before_dedup = len(sediment_data)
# air_data     = deduplicate_and_count_sources(air_data).reset_index(drop=True)
# soil_data    = deduplicate_and_count_sources(soil_data).reset_index(drop=True)
# water_data   = deduplicate_and_count_sources(water_data).reset_index(drop=True)
# sediment_data= deduplicate_and_count_sources(sediment_data).reset_index(drop=True)
# print(f"Air data: {air_before_dedup} entries before deduplication, {len(air_data)} after deduplication.")
# print(f"Soil data: {soil_before_dedup} entries before deduplication, {len(soil_data)} after deduplication.")
# print(f"Water data: {water_before_dedup} entries before deduplication, {len(water_data)} after deduplication.")
# print(f"Sediment data: {sediment_before_dedup} entries before deduplication, {len(sediment_data)} after deduplication.")

In [48]:
# save of processed data, ready to use for database build (independent of later optional merge)
air_data.to_csv(output_dir / "hsbd_air_t_half_data_excl_missing.csv", index=False, sep="\t")
soil_data.to_csv(output_dir / "hsbd_soil_t_half_data_excl_missing.csv", index=False, sep="\t")
water_data.to_csv(output_dir / "hsbd_water_t_half_data_excl_missing.csv", index=False, sep="\t")
sediment_data.to_csv(output_dir / "hsbd_sediment_t_half_data_excl_missing.csv", index=False, sep="\t")